# TagAI on Kaggle - Azərbaycan dili ilə sıfırdan təlim

**Platform seçimi:** Colab işləmirsə Kaggle alternativdir. Eyni `tagai` paketini işlədir.

**Bu nət notebook nədir?** TagAI-nin təlim boru xəttini (data -> tokenizer -> model -> təlim -> qiymətləndirmə) bir səhifədə icra edir.

**What this is.** The complete TagAI pipeline in one notebook: data -> tokenizer -> model -> training -> evaluation -> export.

**Nə vacibdir:** Heç bir əvvəlcədən təlim edilmiş model yüklənmir.
**Critical:** No pretrained weights are downloaded or loaded. Every trainable parameter is randomly initialised.

## 0. Qısa yol / Fast path: əvvəlcədən hazırlanmış model yükləmək

Colab və ya Kaggle-da özünüz əvvəlcədən bir `tagai_run.zip` ixrac etmisinizsə, aşağıdakı hüceyrələri işlədin və təlimi səhifə 5-dən başlayın.
If you exported `tagai_run.zip` from a previous session, upload it with the cell below and skip straight to training.

In [ ]:
#@title 0. Fayl yükləmə (varsa) / Upload an existing run (optional)
from pathlib import Path
import zipfile, shutil, os

UPLOAD = False  # True olduqda "tagai_run.zip" faylını seçin.
RUN_DIR = Path("/kaggle/working/tagai")

if UPLOAD:
    from kaggle_secrets import Environment
    print("Colab-da fayl yükləmək üçün files.upload() istifadə edin.")
    print("On Kaggle use the 'Add Input' panel or files.upload() in Colab.")

if (Path("/kaggle/working") / "tagai_run.zip").exists():
    with zipfile.ZipFile("/kaggle/working/tagai_run.zip") as z:
        z.extractall("/kaggle/working")
    print("extracted an existing run")

## 1. Mühit və GPU / Environment and GPU

In [ ]:
#@title 1. Environment
import os, sys, platform, subprocess, json, time
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.run(["nvidia-smi"], check=False)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", DEVICE)
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory/1024**3:.1f} GB")
else:
    print("GPU tapilmadi / no GPU found")

ROOT = Path("/kaggle/working/tagai") if "kaggle" in sys.modules else Path("/content/tagai_runs")
print("run root:", ROOT)

## 2. Asılılıqlar / Dependencies

In [ ]:
#@title 2. Dependencies
!pip -q install -q "torch>=2.1" "sentencepiece>=0.1.99" "numpy>=1.24" "tqdm>=4.65" "langdetect>=1.0.9" "safetensors>=0.4.0" "matplotlib>=3.7"
print("installed")

## 3. Mənbə kodu / Load the TagAI source

In [ ]:
#@title 3. Source
import subprocess, sys
from pathlib import Path

USER = "Tagiyevff"
REPO = "tagiai"
WORKDIR = ROOT

if not (WORKDIR / "tagai" / "model" / "transformer.py").exists():
    r = subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{USER}/{REPO}.git", str(WORKDIR)],
                       capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")

sys.path.insert(0, str(WORKDIR))
import tagai
print("tagai", tagai.__version__)

## 4. Məlumat / Data

In [ ]:
#@title 4. Data
from tagai.config import build_config
from tagai.data.download import download_file, list_wiki_dump_files, extract_wiki_streams, read_multistream_index

RAW = {
    "paths": {"root": str(ROOT / "main")},
    "data": {"sources": ["azwiki"], "min_chars": 250, "validation_fraction": 0.002, "seed": 1337},
    "tokenizer": {"vocab_size": 32000, "num_threads": 4},
    "model": {"preset": "tagai-base", "max_seq_len": 512},
    "train": {"preset": "main", "micro_batch_size": 8, "grad_accum_steps": 8,
              "max_seq_len": 512, "max_steps": 60000, "resume": "auto"},
}
cfg = build_config(RAW)
paths = cfg["paths"]; paths.ensure()

base = "https://dumps.wikimedia.org/azwiki/latest/"
files = list_wiki_dump_files(base)
idx = next(f for f in files if "multistream-index" in f["name"])
dump = next(f for f in files if "pages-articles-multistream.xml.bz2" in f["name"])
idx_path = paths.raw / "azwiki" / "pages-articles-multistream-index.txt.bz2"
download_file(base + idx["url"], idx_path)
offsets = read_multistream_index(idx_path)
xml = extract_wiki_streams(base + dump["url"], offsets, max_streams=25)
sample = paths.raw / "azwiki" / "sample_articles.xml"
sample.write_text(xml, encoding="utf-8")
print("wrote", sample, xml.count("<page>"), "pages")

In [ ]:
#@title 4b. Clean, tokenize, pack
from tagai.data.prepare import build_corpus, iter_corpus
from tagai.tokenizer.trainer import train_tokenizer
from tagai.tokenizer import TagAITokenizer
from tagai.data.pack import pack_corpus

manifest = build_corpus(paths, cfg["data"], progress_every=5000)
print(json.dumps(manifest["counts"], indent=2))

texts = [r["text"] for r in iter_corpus(paths, "train")]
print("tokenizer corpus:", len(texts), "documents")
tok_stats = train_tokenizer(texts, paths.tokenizer_dir, cfg["tokenizer"])
print("vocab", tok_stats.vocab_size, "alphabet coverage", tok_stats.vocabulary["az_alphabet_coverage"])

tok = TagAITokenizer.from_pretrained(paths.tokenizer_dir)
for split in ("train", "validation"):
    _, st = pack_corpus(paths, tok, split, max_doc_tokens=2048)
    print(split, st.to_dict())

## 5. Model və təlim / Model and training

In [ ]:
#@title 5. Train
from dataclasses import asdict
import torch
from tagai.config import ModelConfig
from tagai.model import TagAIModel
from tagai.data.pack import BatchSampler, PackedDataset
from tagai.training.trainer import Trainer
from tagai.utils import read_json, set_seed

train_cfg = cfg["train"]
if DEVICE == "cuda":
    gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    train_cfg.micro_batch_size = 8 if gb < 20 else 16
print("micro_batch_size:", train_cfg.micro_batch_size)

model_cfg = ModelConfig(**{**asdict(cfg["model"]), "vocab_size": tok.vocab_size})
set_seed(train_cfg.seed)
model = TagAIModel(model_cfg)
print("parameters:", model.num_parameters())

train_meta = read_json(paths.processed / "tokens" / "train_meta.json")
val_meta_path = paths.processed / "tokens" / "validation_meta.json"
train_ds = PackedDataset(train_meta, train_cfg.max_seq_len)
val_ds = PackedDataset(read_json(val_meta_path), train_cfg.max_seq_len) if val_meta_path.exists() else None
train_sampler = BatchSampler(train_ds, train_cfg.micro_batch_size, seed=train_cfg.seed)

run_dir = ROOT / "main"
trainer = Trainer(model, train_sampler, val_ds, train_cfg, model_cfg, run_dir, tokenizer=tok)
rp = trainer.resolve_resume_path()
if rp:
    print("resuming from", rp); trainer.resume(rp)
else:
    print("RANDOM initialization / təsadüfi başlatma")

state = trainer.train()
trainer.save()
print(json.dumps({k: v for k, v in state.to_dict().items() if k != "history"}, indent=2, default=str))

## 6. Nəticələr, ixrac / Results and export

In [ ]:
#@title 6. Evaluate and export
from tagai.inference import TagAIGenerator, GenerateConfig
from tagai.evaluation import EvalSuite
from tagai.export import export_bundle
from tagai.training.checkpoint import find_latest_valid
from tagai.utils import atomic_write_json

ckpt = find_latest_valid(run_dir)
gen = TagAIGenerator(ckpt, paths.tokenizer_dir, device=DEVICE)
for p in ["Salam!", "Azərbaycan haqqında məlumat ver.", "Bir gün Bakıda gəzirdim və"]:
    print(f"> {p}\n{gen.generate(p, GenerateConfig(max_new_tokens=64, seed=1337))}\n")

rep = EvalSuite(max_new_tokens=64).run(gen)
atomic_write_json(paths.reports / "evaluation.json", rep)
print(json.dumps({k: v for k, v in rep.items() if k != "results"}, indent=2, ensure_ascii=False))

bundle = export_bundle(ckpt, paths.tokenizer_dir, paths.export,
                       evaluation_report=rep)
print(json.dumps({k: bundle[k] for k in ("name", "weights_human", "archive", "pretrained_weights_used")},
                 indent=2, ensure_ascii=False))
print("\nKaggle-də çıxışı 'Save Version' -> 'Save & Run All' ilə yükləyə bilərsiniz.")
print("On Kaggle use 'Save Version' with 'Save & Run All' to keep the output files.")

## Yekun / Summary

Kaggle alternatividir: eyni `tagai` paketi, eyni model kodu, eyni tokenizer konfiqurasiyası.
Platform fərqləri (GPU növü, RAM, disk) avtomatik aşkarlanır və batch ölçüsü ona görə seçilir.